# 03 — Semantic Embeddings

## Purpose

Convert extracted project components and subcomponents into semantic embeddings
for innovation-oriented retrieval.

This notebook:

1. Loads extracted analytical units.
2. Splits long text into overlapping chunks.
3. Generates normalized sentence embeddings.
4. Saves chunk metadata and embedding arrays for downstream retrieval.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

ROOT = Path("..").resolve()

PROCESSED_DIR = ROOT / "data" / "processed"

UNITS_PATH = PROCESSED_DIR / "pad_components_subcomponents.csv"
CHUNK_METADATA_PATH = PROCESSED_DIR / "pad_chunk_metadata.csv"
EMBEDDINGS_PATH = PROCESSED_DIR / "pad_chunk_embeddings.npy"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

CHUNK_WORDS = 180
CHUNK_OVERLAP = 30
MIN_CHUNK_WORDS = 30

In [ ]:
# --------------------------------------------------
# LOAD EXTRACTED ANALYTICAL UNITS
# --------------------------------------------------

units_df = pd.read_csv(UNITS_PATH)

print("Analytical units:", len(units_df))
print()
print(units_df["character_count"].describe().to_string())

In [ ]:
# --------------------------------------------------
# CHUNK AND BUILD EMBEDDING TEXT
# --------------------------------------------------

def chunk_text(text, chunk_words=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    """Split text into overlapping word-based chunks."""
    if pd.isna(text):
        return []
    words = str(text).split()
    if not words:
        return []
    chunks, start = [], 0
    while start < len(words):
        end = start + chunk_words
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start += chunk_words - overlap
    return chunks


chunk_records = []
for row_idx, row in units_df.iterrows():
    for chunk_id, chunk in enumerate(chunk_text(row["text"])):
        chunk_records.append({
            "unit_id":    row_idx,
            "projectid":  row["projectid"],
            "type":       row["type"],
            "number":     row["number"],
            "title":      row["title"],
            "chunk_id":   chunk_id,
            "chunk_text": chunk,
        })

chunks_df = pd.DataFrame(chunk_records)
chunks_df["chunk_words"] = chunks_df["chunk_text"].str.split().str.len()

# Remove very short residual chunks
chunks_df = chunks_df[
    chunks_df["chunk_words"] >= MIN_CHUNK_WORDS
].reset_index(drop=True)

chunks_df["embedding_text"] = (
    "Project unit: "
    + chunks_df["type"].astype(str) + " "
    + chunks_df["number"].astype(str)
    + "\nTitle: " + chunks_df["title"].fillna("").astype(str)
    + "\nText: "  + chunks_df["chunk_text"]
)

print("Analytical units:", len(units_df))
print("Chunks retained :", len(chunks_df))
print()
print(chunks_df["chunk_words"].describe().to_string())

In [ ]:
# --------------------------------------------------
# GENERATE EMBEDDINGS
# --------------------------------------------------

model = SentenceTransformer(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

embeddings = model.encode(
    chunks_df["embedding_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Embedding matrix shape:", embeddings.shape)
print("First vector norm      :", round(float(np.linalg.norm(embeddings[0])), 6))

In [ ]:
# --------------------------------------------------
# SAVE
# --------------------------------------------------

np.save(PROCESSED_DIR / "pad_chunk_embeddings.npy", embeddings)
chunks_df.to_csv(PROCESSED_DIR / "pad_chunk_metadata.csv", index=False)

print("Saved:")
print("  pad_chunk_embeddings.npy")
print("  pad_chunk_metadata.csv")